# End-Semester Deep Learning Project (22AI304) — Colab Pro Setup & E01 (Original PUAD)

**Team 11** | Benchmark: **MVTec LOCO AD (Logical Anomalies Only)** | Primary Metric: **Combined Mean I-AUROC** | Secondary Metric: **Pooled Raw I-AUROC**

This notebook performs:
1. **Section 3 Mandatory Hardware & Runtime Audit** (CUDA, GPU model, VRAM, PyTorch/CUDA, RAM, Disk — never assuming A100).
2. **Google Drive Mount & Persistent Storage Setup**.
3. **Code Sync (GitHub or Drive)** and fast extraction of datasets to Colab local SSD (`/content/data/`).
4. **Full 7-Point Verification** via `scripts/verify_env.py`.
5. **E01 Unified Control Run** (`Original PUAD`, `batch_size=8`, `50 epochs`, `BF16`, auto-resuming & Drive-synced).

In [ ]:
#@title 1. Section 3 — Hardware, Runtime, RAM & Disk Audit
import os, platform, shutil, subprocess, sys, torch

print('=== COLAB SESSION AUDIT ===')
print('Python        :', sys.version.split()[0])
print('Platform      :', platform.platform())
print('PyTorch       :', torch.__version__)
print('CUDA Version  :', torch.version.cuda)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU Model     : {props.name}')
    print(f'GPU VRAM      : {props.total_memory / (1024**3):.2f} GiB (CC {props.major}.{props.minor})')
    print(f'BF16 Supported: {torch.cuda.is_bf16_supported()}')
else:
    raise RuntimeError('No GPU detected! Switch Runtime -> Change runtime type -> A100 GPU.')

try:
    import psutil
    vm = psutil.virtual_memory()
    print(f'System RAM    : {vm.available/(1024**3):.1f} GB free / {vm.total/(1024**3):.1f} GB total')
except ImportError:
    pass

disk = shutil.disk_usage('/content')
print(f'Local SSD     : {disk.free/(1024**3):.1f} GB free / {disk.total/(1024**3):.1f} GB total')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
#@title 2. Mount Google Drive & Configure Paths
from google.colab import drive
drive.mount('/content/drive')

# ---- EDIT THESE PATHS TO MATCH YOUR GOOGLE DRIVE ----
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/DL_Endsem_Team11'
DRIVE_LOCO_TAR    = f'{DRIVE_PROJECT_DIR}/data/mvtec_loco_anomaly_detection.tar.xz'
DRIVE_IMNET_ZIP   = f'{DRIVE_PROJECT_DIR}/data/imagenet_penalty_10k.zip'
DRIVE_TEACHER_PT  = f'{DRIVE_PROJECT_DIR}/weights/teacher_pdn_s.pt'
DRIVE_RESULTS_DIR = f'{DRIVE_PROJECT_DIR}/runs'
GITHUB_REPO_URL   = 'https://github.com/Aneesh241/dl-endsem-logical-ad.git'

import os
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
for label, p in [('LOCO tar.xz', DRIVE_LOCO_TAR), ('ImageNet zip', DRIVE_IMNET_ZIP), ('Teacher pt', DRIVE_TEACHER_PT)]:
    print(f'{label:<15}: {"FOUND" if os.path.exists(p) else "MISSING"} -> {p}')

In [ ]:
#@title 3. Clone/Sync Code & Install Dependencies
import os, subprocess

REPO_DIR = '/content/Endsem'
if GITHUB_REPO_URL:
    if os.path.exists(REPO_DIR):
        subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    else:
        subprocess.run(['git', 'clone', GITHUB_REPO_URL, REPO_DIR], check=True)
else:
    # Fallback: copy code folder from Google Drive
    drive_code = f'{DRIVE_PROJECT_DIR}/Endsem'
    assert os.path.isdir(drive_code), f'Provide GITHUB_REPO_URL or upload Endsem/ to {drive_code}'
    subprocess.run(['rsync', '-a', '--exclude', 'checkpoints', '--exclude', 'data', drive_code + '/', REPO_DIR + '/'], check=True)

!pip install -q timm thop einops pyyaml psutil tqdm
%cd /content/Endsem

In [ ]:
#@title 4. Unpack Datasets to Fast Local SSD (/content/data/) — Never Train Over Drive FUSE
import os

LOCAL_LOCO = '/content/data/mvtec_loco_anomaly_detection'
LOCAL_IMNET = '/content/data/imagenet_penalty'
os.makedirs('/content/data', exist_ok=True)

if not os.path.isdir(os.path.join(LOCAL_LOCO, 'breakfast_box')):
    print('Extracting MVTec LOCO AD to local SSD...')
    os.makedirs(LOCAL_LOCO, exist_ok=True)
    !tar -xf "$DRIVE_LOCO_TAR" -C "$LOCAL_LOCO"
else:
    print('MVTec LOCO AD already present on local SSD.')

if not os.path.isdir(LOCAL_IMNET):
    print('Extracting ImageNet penalty corpus to local SSD...')
    !unzip -q "$DRIVE_IMNET_ZIP" -d /content/data/
else:
    print('ImageNet penalty corpus already present on local SSD.')

os.environ['ENDSEM_LOCO_ROOT'] = LOCAL_LOCO
os.environ['ENDSEM_IMAGENET_ROOT'] = LOCAL_IMNET
os.environ['ENDSEM_TEACHER_PATH'] = DRIVE_TEACHER_PT

!python scripts/verify_env.py --save-json "$DRIVE_RESULTS_DIR/colab_env_audit.json"

In [ ]:
#@title 5. Run E01 (Original PUAD — Unified A100 Protocol, Auto-Resumable & Drive-Synced)
import torch
amp_mode = 'bf16' if torch.cuda.is_bf16_supported() else 'fp16'
print(f'Selected AMP mode based on verified GPU: {amp_mode}')

!python scripts/run_experiment.py \
    --config configs/E01_puad.yaml \
    --mode train \
    --mixed-precision $amp_mode \
    --drive-sync-dir "$DRIVE_RESULTS_DIR"